In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, lit, current_timestamp,
    sha2, concat_ws, coalesce
)

# ==========================================
# 05 - GENERIC SCD PROCESSING
# ==========================================

configs = (
    spark.table("mortgage_catalog.metadata.silver_config")
    .filter("is_active = true AND scd_type IS NOT NULL")
    .collect()
)

for config in configs:

    source_table = config["target_table"]       # Clean Silver
    target_table = config["scd_target_table"]
    primary_key  = config["primary_key"]
    scd_type     = config["scd_type"]

    source = f"mortgage_catalog.silver.{source_table}"
    target = f"mortgage_catalog.silver.{target_table}"

    print(f"\n{scd_type}: {source} → {target}")

    if not spark.catalog.tableExists(source):
        print(f"SKIPPED → {source} not found")
        continue

    source_df = spark.table(source)

    # ======================================
    # SCD TYPE 1
    # ======================================

    if scd_type == "SCD1":

        if not spark.catalog.tableExists(target):

            (
                source_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target)
            )

            print(f"SCD1 INITIAL LOAD → {target}")

        else:

            delta_target = DeltaTable.forName(spark, target)

            (
                delta_target.alias("T")
                .merge(
                    source_df.alias("S"),
                    f"T.{primary_key} = S.{primary_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            print(f"SCD1 MERGE → {target}")

    # ======================================
    # SCD TYPE 2
    # ======================================

    elif scd_type == "SCD2":

        technical_cols = {
            primary_key,
            "dq_status",
            "dq_error",
            "_ingestion_timestamp",
            "_source_system",
            "_source_type"
        }

        compare_cols = [
            c for c in source_df.columns
            if c not in technical_cols
        ]

        # Create hash for change detection
        source_df = source_df.withColumn(
            "_record_hash",
            sha2(
                concat_ws(
                    "||",
                    *[
                        coalesce(
                            col(c).cast("string"),
                            lit("")
                        )
                        for c in compare_cols
                    ]
                ),
                256
            )
        )

        # ----------------------------------
        # Initial SCD2 load
        # ----------------------------------

        if not spark.catalog.tableExists(target):

            initial_df = (
                source_df
                .withColumn(
                    "effective_from",
                    current_timestamp()
                )
                .withColumn(
                    "effective_to",
                    lit(None).cast("timestamp")
                )
                .withColumn(
                    "is_current",
                    lit(True)
                )
            )

            (
                initial_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target)
            )

            print(f"SCD2 INITIAL LOAD → {target}")
            continue

        # ----------------------------------
        # Find changed/new records
        # ----------------------------------

        current_df = (
            spark.table(target)
            .filter(col("is_current") == True)
            .select(
                col(primary_key),
                col("_record_hash").alias("target_hash")
            )
        )

        changes = (
            source_df.alias("S")
            .join(
                current_df.alias("T"),
                primary_key,
                "left"
            )
            .filter(
                col("T.target_hash").isNull()
                |
                (
                    col("S._record_hash")
                    != col("T.target_hash")
                )
            )
            .select("S.*")
        )

        if changes.limit(1).count() == 0:
            print(f"NO CHANGES → {target}")
            continue

        # ----------------------------------
        # Expire old records
        # ----------------------------------

        delta_target = DeltaTable.forName(
            spark,
            target
        )

        (
            delta_target.alias("T")
            .merge(
                changes.alias("S"),
                f"""
                T.{primary_key} = S.{primary_key}
                AND T.is_current = true
                """
            )
            .whenMatchedUpdate(
                set={
                    "is_current": "false",
                    "effective_to": "current_timestamp()"
                }
            )
            .execute()
        )

        # ----------------------------------
        # Insert new versions
        # ----------------------------------

        new_versions = (
            changes
            .withColumn(
                "effective_from",
                current_timestamp()
            )
            .withColumn(
                "effective_to",
                lit(None).cast("timestamp")
            )
            .withColumn(
                "is_current",
                lit(True)
            )
        )

        (
            new_versions.write
            .format("delta")
            .mode("append")
            .saveAsTable(target)
        )

        print(f"SCD2 COMPLETE → {target}")

print("\nALL SCD PROCESSING COMPLETED ✅")


SCD1: mortgage_catalog.silver.property_clean → mortgage_catalog.silver.property_current
SCD1 MERGE → mortgage_catalog.silver.property_current

SCD2: mortgage_catalog.silver.branch_clean → mortgage_catalog.silver.branch_history
SCD2 COMPLETE → mortgage_catalog.silver.branch_history

SCD1: mortgage_catalog.silver.document_clean → mortgage_catalog.silver.document_current
SCD1 MERGE → mortgage_catalog.silver.document_current

SCD1: mortgage_catalog.silver.loan_application_clean → mortgage_catalog.silver.loan_application_current
SCD1 MERGE → mortgage_catalog.silver.loan_application_current

SCD1: mortgage_catalog.silver.payment_clean → mortgage_catalog.silver.payment_current
SCD1 MERGE → mortgage_catalog.silver.payment_current

SCD1: mortgage_catalog.silver.credit_score_clean → mortgage_catalog.silver.credit_score_current
SCD1 MERGE → mortgage_catalog.silver.credit_score_current

SCD1: mortgage_catalog.silver.customer_clean → mortgage_catalog.silver.customer_current
SCD1 MERGE → mortgage_c